## Problem 3: How far did people travel? (8 points)

During this task, the aim is to calculate the (air-line) distance in meters that each social media user in the data set prepared in *Problem 2* has travelled in-between the posts. We’re interested in the Euclidean distance between subsequent points generated by the same user.

For this, we will need to use the `userid` column of the data set `kruger_posts.shp` that we created in *Problem 2*.

Answer the following questions:
- What was the shortest distance a user travelled between all their posts (in meters)?
- What was the mean distance travelled per user (in meters)?
- What was the maximum distance a user travelled (in meters)?

---


### a) Read the input file and re-project it

- Read the input file `kruger_points.shp` into a geo-data frame `kruger_points`
- Transform the data from WGS84 to an `EPSG:32735` projection (UTM Zone 35S, suitable for South Africa). This CRS has *metres* as units.

In [1]:
# Import libraries
import pathlib
import geopandas as gpd

# Define Data directory
DATA_DIRECTORY = pathlib.Path().resolve() / "data"

# Read file
kruger_points = gpd.read_file(DATA_DIRECTORY / "kruger_points.shp")

# Transform the data from WGS84 to an EPSG:32735 projection (
kruger_points = kruger_points.to_crs(epsg=32735)

In [2]:
# NON-EDITABLE CODE CELL FOR TESTING YOUR SOLUTION

# Check the data
kruger_points.head()

,lat,lon,timestamp,userid,geometry
0,-24.980792,31.484633,2015-07-07 03:02,66487960,POINT (952912.890 7229683.258)
1,-25.499225,31.508906,2015-07-07 03:18,65281761,POINT (953433.223 7172080.632)
2,-24.342578,30.930866,2015-03-07 03:38,90916112,POINT (898955.144 7302197.408)
3,-24.854614,31.519718,2015-10-07 05:04,37959089,POINT (956927.218 7243564.942)
4,-24.921069,31.520836,2015-10-07 05:19,27793716,POINT (956794.955 7236187.926)


In [3]:
# NON-EDITABLE CODE CELL FOR TESTING YOUR SOLUTION

# Check that the crs is correct after re-projecting (should be epsg:32735)
import pyproj
assert kruger_points.crs == pyproj.CRS("EPSG:32735")

### b) Group the data by user id

Group the data by `userid` and store the grouped data in a variable `grouped_by_users`

In [4]:
# ADD YOUR OWN CODE HERE

In [5]:
# Group de data
grouped_by_users = kruger_points.groupby(["userid"])
grouped_by_users.head()

,lat,lon,timestamp,userid,geometry
0,-24.980792,31.484633,2015-07-07 03:02,66487960,POINT (952912.890 7229683.258)
1,-25.499225,31.508906,2015-07-07 03:18,65281761,POINT (953433.223 7172080.632)
2,-24.342578,30.930866,2015-03-07 03:38,90916112,POINT (898955.144 7302197.408)
3,-24.854614,31.519718,2015-10-07 05:04,37959089,POINT (956927.218 7243564.942)
4,-24.921069,31.520836,2015-10-07 05:19,27793716,POINT (956794.955 7236187.926)
...,...,...,...,...,...
81361,-25.431166,31.952533,2015-11-04 20:38,120615,POINT (998394.873 7178043.198)
81368,-25.385833,31.828889,2015-04-05 01:10,5970719,POINT (986115.441 7183532.368)
81372,-25.080739,31.918978,2015-08-05 01:52,46410906,POINT (996441.120 7217063.910)
81374,-24.799541,31.354469,2015-09-05 02:23,90744213,POINT (940395.058 7250220.045)


In [6]:
# NON-EDITABLE CODE CELL FOR TESTING YOUR SOLUTION

# Check the number of groups:
assert len(grouped_by_users.groups) == kruger_points["userid"].nunique(), "Number of groups should match number of unique users!"

### c) Create `shapely.geometry.LineString` objects for each user connecting the points from oldest to most recent

There are multiple ways to solve this problem (see the [hints for this exercise](https://autogis-site.readthedocs.io/en/latest/lessons/lesson-2/exercise-2.html). You can use, for instance, a dictionary or an empty GeoDataFrame to collect data that is generated using the steps below:

- Use a for-loop to iterate over the grouped object. For each user’s data: 
    - [sort](http://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.sort_values.html) the rows by timestamp 
    - create a `shapely.geometry.LineString` based on the user’s points

**CAREFUL**: Remember that every LineString needs at least two points. Skip users who have less than two posts.

Store the results in a `geopandas.GeoDataFrame` called `movements`, and remember to assign a CRS.

In [9]:
# Import libraries
from shapely.geometry import LineString

# Create an empty dictionnary
movements = {
    "userid": [],
    "geometry": []
}

# For-loop to iterate over the grouped object
for userid, group in grouped_by_users:
    
    # Sort rows by timestamp
    group = group.sort_values("timestamp")
    
    # Skip users who have less than two points
    if len(group) < 2:
        continue
    
    # Create LineString
    line = LineString(group.geometry.tolist())
    
    movements["userid"].append(userid)
    movements["geometry"].append(line)

# Store results
movements = gpd.GeoDataFrame(movements, crs = "EPSG:32735")

/tmp/ipykernel_461/2204849602.py:11: FutureWarning: In a future version of pandas, a length 1 tuple will be returned when iterating over a groupby with a grouper equal to a list of length 1. Don't supply a list with a single grouper to avoid this warning.
  for userid, group in grouped_by_users:


In [10]:
# NON-EDITABLE CODE CELL FOR TESTING YOUR SOLUTION

# Check the result
print(type(movements))
print(movements.crs)

movements

<class 'geopandas.geodataframe.GeoDataFrame'>
EPSG:32735


,userid,geometry
0,16301,"LINESTRING (942231.630 7254606.868, 938934.725..."
1,45136,"LINESTRING (905394.500 7193375.148, 905394.500..."
2,50136,"LINESTRING (944551.607 7253384.183, 963788.403..."
3,88775,"LINESTRING (902800.817 7192546.975, 902800.839..."
4,88918,"LINESTRING (959332.961 7219877.715, 963788.403..."
...,...,...
9021,99921781,"LINESTRING (902885.190 7196931.096, 904027.710..."
9022,99936874,"LINESTRING (963782.211 7228000.079, 963754.402..."
9023,99964140,"LINESTRING (938876.653 7305143.369, 938876.943..."
9024,99986933,"LINESTRING (935937.029 7305973.536, 936598.681..."


### d) Calculate the distance between all posts of a user

- Check once more that the CRS of the data frame is correct
- Compute the lengths of the lines, and store it in a new column called `distance`

In [13]:
# Check that the CRS is correct
assert movements.crs.to_epsg() == 32735

# Store the length of lines into a new columns named distance
movements["distance"] = movements.geometry.length

In [14]:
# NON-EDITABLE CODE CELL FOR TESTING YOUR SOLUTION

#Check the output
movements.head()

,userid,geometry,distance
0,16301,"LINESTRING (942231.630 7254606.868, 938934.725...",328455.115430
1,45136,"LINESTRING (905394.500 7193375.148, 905394.500...",0.000000
2,50136,"LINESTRING (944551.607 7253384.183, 963788.403...",159189.081019
3,88775,"LINESTRING (902800.817 7192546.975, 902800.839...",0.080245
4,88918,"LINESTRING (959332.961 7219877.715, 963788.403...",9277.252211


### e) Answer the original questions

You should now be able to quickly find answers to the following questions: 
- What was the shortest distance a user travelled between all their posts (in meters)? (store the value in a variable `shortest_distance`)
- What was the mean distance travelled per user (in meters)? (store the value in a variable `mean_distance`)
- What was the maximum distance a user travelled (in meters)? (store the value in a variable `longest_distance`)

In [16]:
shortest_distance = movements["distance"].min()
mean_distance = movements["distance"].mean()
longest_distance = movements["distance"].max()

shortest_distance, mean_distance, longest_distance


(0.0, 107133.51202944393, 6970668.816343962)

### f) Save the movements in a file

Save the `movements` into a new Shapefile called `movements.shp` inside the `data` directory.

In [18]:
# Save shapefile
movements.to_file(DATA_DIRECTORY / "movements.shp")

In [19]:
# NON-EDITABLE CODE CELL FOR TESTING YOUR SOLUTION

assert (DATA_DIRECTORY / "movements.shp").exists()


---

# Fantastic job!

That’s all for this week! 